In [2]:
import pandas as pd

df = pd.read_csv('creditcard.csv')
print("Shape:", df.shape)
df.head()


Shape: (284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
print(df['Class'].value_counts())
print("\nFraud %:", round(df['Class'].astype(int).mean()*100, 3), "%")

Class
0    284315
1       492
Name: count, dtype: int64

Fraud %: 0.173 %


In [4]:
print(df.isnull().sum().sum())  # should be 0

0


In [5]:
df[['Amount']].describe()

,Amount
count,284807.000000
mean,88.349619
std,250.120109
min,0.000000
25%,5.600000
50%,22.000000
75%,77.165000
max,25691.160000


In [6]:
df.groupby('Class')[['Amount']].mean()

,Amount
Class,
0,88.291022
1,122.211321


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Separate features and target
X = df.drop('Class', axis=1)
y = df['Class']

# Split into train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train baseline model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# Predict
y_pred = model.predict(X_test_scaled)

# Evaluate
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[56851    13]
 [   36    62]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.83      0.63      0.72        98

    accuracy                           1.00     56962
   macro avg       0.91      0.82      0.86     56962
weighted avg       1.00      1.00      1.00     56962



In [8]:
baseline_results = {
    'model': 'Baseline Logistic Regression (default threshold 0.5)',
    'accuracy': (56851+62)/56962,
    'fraud_caught': 62,
    'fraud_missed': 36,
    'false_alarms': 13
}
print(baseline_results)

{'model': 'Baseline Logistic Regression (default threshold 0.5)', 'accuracy': 0.9991397773954567, 'fraud_caught': 62, 'fraud_missed': 36, 'false_alarms': 13}


In [12]:
# Train cost-sensitive model — tells the model fraud errors matter more
cost_sensitive_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'   # this is the key change from baseline
)
cost_sensitive_model.fit(X_train_scaled, y_train)

y_pred_cs = cost_sensitive_model.predict(X_test_scaled)

print(confusion_matrix(y_test, y_pred_cs))
print(classification_report(y_test, y_pred_cs))

[[55478  1386]
 [    8    90]]
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.98     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.98      0.99     56962



In [13]:
FP_cost = 25
FN_cost = 500

# Get probabilities instead of hard predictions
y_proba_cs = cost_sensitive_model.predict_proba(X_test_scaled)[:, 1]

import numpy as np

best_threshold = 0.5
best_cost = float('inf')

for threshold in np.arange(0.01, 1.0, 0.01):
    y_pred_t = (y_proba_cs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
    total_cost = fp * FP_cost + fn * FN_cost
    if total_cost < best_cost:
        best_cost = total_cost
        best_threshold = threshold

print(f"Best threshold: {best_threshold:.2f}")
print(f"Minimum total cost: ${best_cost:,}")

Best threshold: 0.97
Minimum total cost: $7,875


In [14]:
y_pred_optimal = (y_proba_cs >= best_threshold).astype(int)
print(confusion_matrix(y_test, y_pred_optimal))
print(classification_report(y_test, y_pred_optimal))

[[56769    95]
 [   11    87]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.48      0.89      0.62        98

    accuracy                           1.00     56962
   macro avg       0.74      0.94      0.81     56962
weighted avg       1.00      1.00      1.00     56962



In [15]:
final_comparison = {
    'baseline': {'threshold': 0.5, 'fraud_caught': 62, 'fraud_missed': 36, 'false_alarms': 13, 'total_cost': 18325},
    'balanced': {'threshold': 0.5, 'fraud_caught': 90, 'fraud_missed': 8, 'false_alarms': 1386, 'total_cost': 38650},
    'optimized': {'threshold': 0.97, 'fraud_caught': 87, 'fraud_missed': 11, 'false_alarms': 95, 'total_cost': 7875}
}

import json
with open('results_summary.json', 'w') as f:
    json.dump(final_comparison, f, indent=2)

print("Saved!")
print(f"\nCost reduction vs baseline: {(18325-7875)/18325*100:.1f}%")

Saved!

Cost reduction vs baseline: 57.0%
